# 野生蓝莓产量预测分析## 步骤 5：随机森林

In [ ]:
import pandas as pdimport numpy as npimport sys, ossys.path.insert(0, os.path.join(os.getcwd(), ".."))from src.config import *from src.data.loader import load_trainfrom src.data.preprocessor import DataPreprocessorfrom src.models.random_forest import BlueberryRandomForestfrom src.models.evaluation import regression_metrics, cross_validatefrom src.visualization.plots import *train = load_train()preprocessor = DataPreprocessor()df_clean = preprocessor.clean(train)X, y = preprocessor.split_features_target(df_clean)X_scaled = preprocessor.scale(X, fit=True)X_train, X_val, y_train, y_val = preprocessor.train_val_split(X_scaled, y)print(f"Train: {X_train.shape}, Val: {X_val.shape}")

## 5.1 基础随机森林模型

In [ ]:
rf = BlueberryRandomForest()rf.fit(X_train, y_train, n_estimators=200)y_pred_base = rf.predict(X_val)metrics_base = rf.evaluate(y_val, y_pred_base)print("Base RF (n_estimators=200):")for k, v in metrics_base.items():    print(f"  {k}: {v:.4f}")

## 5.2 交叉验证评估

In [ ]:
from sklearn.ensemble import RandomForestRegressorcv_results = cross_validate(    RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),    X_train, y_train, cv=5)print(f"CV Mean R²: {cv_results['cv_mean_r2']:.4f} (+/- {cv_results['cv_std_r2']:.4f})")print(f"CV Scores: {cv_results['cv_scores']}")

## 5.3 网格搜索调参

In [ ]:
rf.grid_search(    X_train, y_train,    param_grid={        "n_estimators": [100, 200, 300],        "max_depth": [None, 10, 20, 30],        "min_samples_split": [2, 5],        "min_samples_leaf": [1, 2],        "max_features": ["sqrt", "log2"],    },    cv=5,    verbose=1)print(f"Best params: {rf.best_params}")

## 5.4 调参后评估

In [ ]:
y_pred_tuned = rf.predict(X_val)metrics_tuned = rf.evaluate(y_val, y_pred_tuned)print("Tuned RF Metrics:")for k, v in metrics_tuned.items():    print(f"  {k}: {v:.4f}")

## 5.5 特征重要性

In [ ]:
importance_df = rf.get_feature_importance(X_train.columns.tolist())print("Feature Importance (top 10):")print(importance_df.head(10).to_string(index=False))plot_feature_importance(    importance_df["feature"].tolist(),    importance_df["importance"].values)

## 5.6 残差分析

In [ ]:
plot_residuals(y_val.values, y_pred_tuned)plot_actual_vs_predicted(y_val.values, y_pred_tuned, "Random Forest (Tuned)")